In [0]:
import importlib
import sys
from pathlib import Path
from pyspark.sql import functions as F
project_root = str(Path.cwd().resolve().parent)
if project_root not in sys.path:
    sys.path.append(project_root)

import utils.storage_config as storage_config
from pyspark.sql import functions as F
importlib.reload(storage_config)
storage_config.spark = spark
storage_config.configure_storage()

In [0]:
df = (
    spark.read
    .format("delta")
    .load("abfss://bronze@secondstorage89.dfs.core.windows.net/products/")
)

In [0]:
df.show(10)

In [0]:
df.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c)
    for c in df.columns
]).show()

In [0]:
df.filter(
    F.col("product_category_name").isNull()
).select(
    "product_id",
    "product_category_name",
    "product_name_lenght",
    "product_description_lenght",
    "product_photos_qty",
    "product_weight_g",
    "product_length_cm",
    "product_height_cm",
    "product_width_cm"
).show(10, truncate=False)

In [0]:
df.filter(
    F.col("product_category_name").isNull()
).filter(
    F.col("product_name_lenght").isNotNull() |
    F.col("product_description_lenght").isNotNull() |
    F.col("product_photos_qty").isNotNull()
).count()

In [0]:
df.filter(
    F.col("product_weight_g").isNull() |
    F.col("product_length_cm").isNull() |
    F.col("product_height_cm").isNull() |
    F.col("product_width_cm").isNull()
).show(truncate=False)

In [0]:
completely_empty_product = (
    F.col("product_category_name").isNull() &
    F.col("product_name_lenght").isNull() &
    F.col("product_description_lenght").isNull() &
    F.col("product_photos_qty").isNull() &
    F.col("product_weight_g").isNull() &
    F.col("product_length_cm").isNull() &
    F.col("product_height_cm").isNull() &
    F.col("product_width_cm").isNull()
)

In [0]:
quarantine_products = df.filter(completely_empty_product)

silver_products = df.filter(~completely_empty_product)

In [0]:
silver_products.write \
    .format("delta") \
    .mode("overwrite") \
    .save(
        "abfss://silver@secondstorage89.dfs.core.windows.net/products/"
    )